In [1]:
import asyncio
import math
import pandas as pd

In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [ ]:
from ib_insync import *
from ibkr.Class_IBKR_IB import IBKR_IB
ibkr = IBKR_IB(port=7496)

async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

await start_ibkr()

IBKR connected: True


Error 200, reqId 12: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=1.0, right='C', exchange='SMART', currency='USD')
Error 200, reqId 13: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=1.0, right='P', exchange='SMART', currency='USD')
Error 200, reqId 16: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=3.0, right='C', exchange='SMART', currency='USD')
Error 200, reqId 15: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=2.0, right='P', exchange='SMART', currency='USD')
Error 200, reqId 17: No security definition has been found for the request, contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=3.0, right='P', exch

In [4]:
stock_symbols = ['IBIT']

futures_symbols = ['BTCM6',
                   'BTCN6',
                   'BTCQ6',
                   'BTCU6',
                   'BTCV6',
                   'BTCX6']

In [5]:
futures_contracts = []

for symbol in futures_symbols:
    contract = Future(localSymbol=symbol, exchange='CME', currency="USD")
    [contract] = await ibkr.ib.qualifyContractsAsync(contract) # this may lead to a printed line since its return has nowhere to be mapped
    futures_contracts.append(contract)
    print(contract)

Future(conId=751356962, symbol='BRR', lastTradeDateOrContractMonth='20260626', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCM6', tradingClass='BTC')
Future(conId=850790355, symbol='BRR', lastTradeDateOrContractMonth='20260731', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCN6', tradingClass='BTC')
Future(conId=859040542, symbol='BRR', lastTradeDateOrContractMonth='20260828', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCQ6', tradingClass='BTC')
Future(conId=772435574, symbol='BRR', lastTradeDateOrContractMonth='20260925', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCU6', tradingClass='BTC')
Future(conId=876880607, symbol='BRR', lastTradeDateOrContractMonth='20261030', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCV6', tradingClass='BTC')
Future(conId=887699043, symbol='BRR', lastTradeDateOrContractMonth='20261127', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCX6', tradingClass

In [6]:
stock_contracts = []
stock_details = []
option_chains = []

for symbol in stock_symbols:
    contract = Stock(symbol, 'SMART', "USD")
    [contract] = await ibkr.ib.qualifyContractsAsync(contract) # this may lead to a printed line since its return has nowhere to be mapped
    stock_contracts.append(contract)
    print(contract)

    [details] = await ibkr.ib.reqContractDetailsAsync(contract)
    stock_details.append(details)
    print(details)

    option_chain = await ibkr.ib.reqSecDefOptParamsAsync(underlyingSymbol=details.contract.symbol,
                                                         futFopExchange="",
                                                         underlyingSecType=details.contract.secType,
                                                         underlyingConId=details.contract.conId
                                                         )
    option_chains.append(option_chain)
    print(option_chain)

    print('count=', len(option_chain[0].expirations), option_chain[0].expirations)
    print('count=', len(option_chain[0].strikes), option_chain[0].strikes)
    print("2 *", len(option_chain[0].expirations), "*", len(option_chain[0].strikes), "=", len(option_chain[0].expirations) * len(option_chain[0].strikes) * 2)

    print('\n')

Stock(conId=677037673, symbol='IBIT', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='IBIT', tradingClass='NMS')
ContractDetails(contract=Contract(secType='STK', conId=677037673, symbol='IBIT', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='IBIT', tradingClass='NMS'), marketName='NMS', minTick=0.01, orderTypes='ACTIVETIM,AD,ADDONT,ADJUST,ALERT,ALGO,ALLOC,AON,AVGCOST,BASKET,BENCHPX,CASHQTY,COND,CONDORDER,DARKONLY,DARKPOLL,DAY,DEACT,DEACTDIS,DEACTEOD,DIS,DUR,GAT,GTC,GTD,GTT,HID,IBKRATS,ICE,IMB,IOC,LIT,LMT,LOC,MIDPX,MIT,MKT,MOC,MTL,NGCOMB,NODARK,NONALGO,OCA,OPG,OPGREROUT,PEGBENCH,PEGMID,POSTATS,POSTONLY,PREOPGRTH,PRICECHK,REL,REL2MID,RELPCTOFS,RPI,RTH,SCALE,SCALEODD,SCALERST,SIZECHK,SNAPMID,SNAPMKT,SNAPREL,STP,STPLMT,SWEEP,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF', validExchanges='SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,ISLAND,DRCTEDGE,BEX,BATS,EDGEA,BYX,IEX,EDGX,FOXRIVER,PEARL,NYSENAT,LTSE,MEMX,IBEOS,OVERNIGHT,TPLUS0,PSX,T24X', priceMagnif

In [7]:
option_contracts = []

for chain, details in zip(option_chains, stock_details):
    for expiry in chain[0].expirations:
        for strike in chain[0].strikes:
            for right in ['C', 'P']:
                option_contract = Option(lastTradeDateOrContractMonth=expiry,
                                         strike=float(strike),
                                         right=right,
                                         symbol=details.contract.symbol,
                                         exchange=details.contract.exchange,
                                         currency=details.contract.currency,
                                        )
                option_contracts.append(option_contract)

option_contracts = await ibkr.ib.qualifyContractsAsync(*option_contracts) # this may lead to printed lines since its return has nowhere to be mapped

print(len(option_contracts))

Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=1.0, right='C', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=1.0, right='P', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=2.0, right='C', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=2.0, right='P', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=3.0, right='C', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=3.0, right='P', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='IBIT', lastTradeDateOrContractMonth='20260622', strike=4.0, right='C', exchange='SMART', currency='USD')
Unknown contract: Option(symbol='I

3042


In [8]:
def clean_num(x):
    """
    Convert IBKR nan / -1 / None style missing values to None.
    """
    if x is None:
        return None
    try:
        if math.isnan(x):
            return None
    except TypeError:
        pass
    if x == -1:
        return None
    return x

def ticker_row(ticker):
    contract = ticker.contract
    return {
            "conId": getattr(ticker.contract, "conId", None),
            "secType": getattr(contract, "secType", None),
            "symbol": getattr(contract, "symbol", None),
            "expiration": getattr(contract, "lastTradeDateOrContractMonth", None),
            "strike": getattr(contract, "strike", None),
            "right": getattr(contract, "right", None),

            "close": clean_num(getattr(ticker, "close", None)),
        #    "volume": clean_num(getattr(ticker, "volume", None)),
            "avgVolume": clean_num(getattr(ticker, "avVolume", None)),
            "futuresOpenInterest": clean_num(getattr(ticker, "futuresOpenInterest", None)),
            "putOpenInterest": clean_num(getattr(ticker, "putOpenInterest", None)),
            "callOpenInterest": clean_num(getattr(ticker, "callOpenInterest", None)),
        }

In [9]:
generic_ticks = "100, 101, 165, 588"

tickers = []

all_contracts = [*futures_contracts, *stock_contracts]#, *option_contracts] 

for contract in all_contracts[:20]:
    print(stock_contracts)
    ticker = ibkr.ib.reqMktData(
                    contract, 
                    genericTickList=generic_ticks,
                    #snapshot=True
                    )
    tickers.append(ticker)

await asyncio.sleep(30)

#   turn off reqMktData

rows = [ticker_row(ticker) for ticker in tickers]

df_all = pd.DataFrame(rows)
df_all

[Stock(conId=677037673, symbol='IBIT', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='IBIT', tradingClass='NMS')]
[Stock(conId=677037673, symbol='IBIT', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='IBIT', tradingClass='NMS')]
[Stock(conId=677037673, symbol='IBIT', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='IBIT', tradingClass='NMS')]
[Stock(conId=677037673, symbol='IBIT', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='IBIT', tradingClass='NMS')]
[Stock(conId=677037673, symbol='IBIT', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='IBIT', tradingClass='NMS')]
[Stock(conId=677037673, symbol='IBIT', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='IBIT', tradingClass='NMS')]
[Stock(conId=677037673, symbol='IBIT', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='IBIT', tradingClass='NMS')]


,conId,secType,symbol,expiration,strike,right,close,avgVolume,futuresOpenInterest,putOpenInterest,callOpenInterest
0,751356962,FUT,BRR,20260626,0.0,,62945.000000,NaN,11899.0,1149.0,695.0
1,850790355,FUT,BRR,20260731,0.0,,63240.000000,NaN,8212.0,246.0,195.0
2,859040542,FUT,BRR,20260828,0.0,,63495.000000,NaN,180.0,16.0,181.0
3,772435574,FUT,BRR,20260925,0.0,,63745.000000,NaN,226.0,149.0,165.0
4,876880607,FUT,BRR,20261030,0.0,,64115.000000,NaN,5.0,26.0,60.0
5,887699043,FUT,BRR,20261127,0.0,,64425.000000,NaN,0.0,0.0,0.0
6,677037673,STK,IBIT,,0.0,,36.360001,440638.0,NaN,0.0,0.0


In [10]:
df_linaer  = df_all[df_all["secType"] != "OPT"].reset_index(drop=True)
df_options = df_all[df_all["secType"] == "OPT"].reset_index(drop=True)

In [11]:
df_option_conIds = (
    df_options.pivot_table(
        index=["symbol", "strike"],
        columns=["expiration", "right"],
        values="conId",
        aggfunc="first"
    )
    .sort_index()
    .sort_index(axis=1)
    .reset_index()
)

df_option_conIds

expiration,symbol,strike
right,,


In [12]:
df_option_prices = (
    df_options.pivot_table(
        index=["symbol", "strike"],
        columns=["expiration", "right"],
        values="close",
        aggfunc="first"
    )
    .sort_index()
    .sort_index(axis=1)
    .reset_index()
)

df_option_prices

expiration,symbol,strike
right,,


In [13]:
# Add parity for each expiration
print(df_option_prices.columns.get_level_values(0).unique())

stock_close = df_linear[df_linear['secType'] == 'STK', 'close'][0]

for expiration in df_option_prices.columns.get_level_values(0).unique():
    call_col = (expiration, "C")
    put_col = (expiration, "P")
    parity_col = (expiration, "parity")
    time_value_col = (expiration, "time_value")
    upfront_combo_flow_col = (expiration, "uprfont_combo_flow")
    expiry_combo_flow_col = (expiration, "expiry_combo_flow")
    combo_return_rate_col = (expiration, "combo_return_rate")


    if call_col in df_option_prices.columns and put_col in df_option_prices.columns:
        df_option_prices[parity_col] = (
            df_option_prices[call_col] - df_option_prices[put_col]
        ).abs()
    
        df_option_prices[time_value_col] = (
            df_option_prices[call_col], df_option_prices[put_col]
        ).min()
    
        df_option_prices[upfront_combo_flow_col] = (
            df_option_prices[put_col] - df_option_prices[call_col] + stock_close
        )

        df_option_prices[expiry_combo_flow_col] = df_option_prices['strike']
    
        df_option_prices[combo_return_rate_col] = (
            df_option_prices[expiry_combo_flow_col] / df_option_prices[upfront_combo_flow_col] - 1
        )
    

# Sort columns again so parity sits with each expiration
df_option_prices = df_option_prices.sort_index(axis=1)

# Bring symbol and strike back as columns
df_option_prices = df_option_prices.reset_index()

df_option_prices

Index(['symbol', 'strike'], dtype='object', name='expiration')


NameError: name 'df_linear' is not defined